# Chapter 16, Part 2 — Live Streaming and Text Analysis with PySpark
**Updated for Spark 3.5+**

### Objectives
* Process static text files efficiently with Spark `DataFrame`s.
* Learn to tokenize, clean, and filter text using Spark ML transformers.
* Implement a real-time streaming analysis pipeline with Structured Streaming.

## Example 1 — Word Count with `DataFrame`s and `StopWordsRemover`
Analyzes *RomeoAndJuliet.txt* using the DataFrame API.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover

### Create `SparkSession`
* Entry point into a Spark application

In [ ]:
spark = (SparkSession.builder
            .appName('RomeoAndJulietCounter')
            .master('local[*]') # run locally using all cores
            .getOrCreate())

### Read text file into `DataFrame` 
* Creates a `DataFrame` with one column named `value` containing each line of text

In [ ]:
lines = spark.read.text('RomeoAndJuliet.txt') 

### Normalize case then strip punctuation
* select each line from the `value` column and lowercase it
* rename (`alias`) the column `line`

In [ ]:
lowered = lines.select(F.lower(F.col('value')).alias('line'))

* replace all punctuation with spaces

In [ ]:
cleaned = lowered.select(F.regexp_replace('line', r'[^a-z0-9]+', ' ').alias('line'))

### Tokenize into arrays
* `RegexTokenizer` takes the `line` column from the previous step as input, then outputs a `tokens` column containing arrays of the individual words 

In [ ]:
regex_tokenizer = RegexTokenizer(inputCol='line', outputCol='tokens', pattern=r'\s+', gaps=True)
tokenized = regex_tokenizer.transform(cleaned)

### Remove stopwords
* `StopWordsRemover` takes the `tokens` column from the previous step and outputs a `tokens_nostop` column with the stop words removed

In [ ]:
remover = StopWordsRemover(inputCol='tokens', outputCol='tokens_nostop')
no_stop_words = remover.transform(tokenized)

### Flatten arrays and filter out empty tokens
* Make every word its own row in a column named `word` and filter out any empty strings

In [ ]:
words = (no_stop_words.select(F.explode('tokens_nostop').alias('word'))
                      .filter((F.length(F.col('word')) > 2) & (F.col('word') != '')))

### Aggregate and sort

In [ ]:
result = (words.groupBy('word')
               .count()
               .filter(F.col('count') >= 60)
               .orderBy(F.desc('count'), F.asc('word')))

In [ ]:
result.show(100, truncate=False)

In [ ]:
spark.stop()


### Notes
- The `RegexTokenizer` splits text into words based on whitespace.
- The `StopWordsRemover` uses Spark's built-in English list.
- You can customize the stopword list if desired.
- The `regexp_replace` expression strips punctuation for consistency.



## Example 2 — Real-Time Hashtag Streaming with Structured Streaming

**New code (Structured Streaming)**
- Uses `DataFrame`s and the unified Spark engine.
- Easier integration with Spark SQL and `DataFrame` operations.
- Supports `outputMode('complete')` for global counts.
- Works seamlessly with `foreachBatch` for custom visualizations.


In [ ]:
import warnings

# disable glyph warnings for missing fonts
warnings.filterwarnings("ignore")

In [ ]:
from pyspark.sql import SparkSession, functions as F
from IPython import display
import matplotlib.pyplot as plt
import seaborn as sns

### Create a `SparkSession`

In [ ]:
spark = (SparkSession.builder
            .appName('MastodonStreaming')
            .master("local[*]") # run locally using all cores
            .getOrCreate())

### Create socket source 
* Socket-based stream of data just as a demo
* Not recommended in production apps

In [ ]:
lines = (spark.readStream
            .format('socket')
            .option('host', 'localhost')
            .option('port', 9876)
            .load())

### Tokenize and filter hashtags
* In `select()` call, `explode()` turns every token into its own `DataFrame` row
* `filter()` call keeps only non-null and non-empty rows

In [ ]:
hashtags = (lines.select(F.explode(F.split(F.col('value'), r'\s+')).alias('hashtag'))
                 .filter((F.col('hashtag').isNotNull()) & (F.col('hashtag') != '')))

### Count occurrences
* Create two-column output of the hashtags and their counts

In [ ]:
hashtag_counts = hashtags.groupBy('hashtag').count()

### Visualization callback
* sort the data
* select the top 20 hashtags 
* convert to Pandas `DataFrame`
* visualize as a Seaborn `barplot`

In [ ]:
def show_top20(batch_df, batch_id):
    # sort and select top 20
    top20_pdf = (
        batch_df.orderBy(F.desc('count'), F.asc('hashtag'))
                .limit(20)
                .toPandas()
    )

    # create the barplot then show it
    display.clear_output(wait=True)
    plt.figure(figsize=(8, 6))
    ax = sns.barplot(data=top20_pdf, x='hashtag', y='count')
    ax.set_title(f'Top Hashtags (batch {batch_id})')
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()

### Clear old checkpoint folder
* Spark apps are normally stateless
* A checkpoint folder maintains state
* As we execute this app, we remove the old checkpoint folder

In [ ]:
import shutil
import os

checkpoint_dir = "/tmp/hashtags_chkpt"

# remove old checkpoint before each run
if os.path.exists(checkpoint_dir):
    shutil.rmtree(checkpoint_dir)
    print(f"Old checkpoint directory '{checkpoint_dir}' removed.")

### Start streaming query
1. `writeStream` returns a `DataStreamWriter` used to configure the app's output
2. `outputMode()` indicates **all results will be output**, not just new/updated ones
3. `option()` specifies checkpoint folder where state information is saved
4. `forEachBatch()` specifies a function to call with the `DataFrame` and a unique ID for the current microbatch of data being processed
5. `trigger()` sets time between batches of data; can be in seconds, minutes, hours or a combo
6. `start()` launches the application

In [ ]:
query = (hashtag_counts.writeStream
            .outputMode('complete')
            .option('checkpointLocation', checkpoint_dir)
            .foreachBatch(show_top20)
            .trigger(processingTime='5 seconds') 
            .start())

### To stop manually in the notebook, run: query.stop()

In [ ]:
query.awaitTermination()